# Experiment Tracking, MLflow, and Model Registry

> Track model runs, compare metrics, and understand versioning beyond saving a single file.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Record parameters, metrics, dataset identifiers, and artifacts for each experiment.
- Understand experiment tracking versus model registry.
- Use a lightweight in-memory run table and see an optional MLflow pattern.

Experiment tracking is the scientific notebook of model development: what changed, which data were used, and whether an improvement is repeatable. MLflow provides APIs and UI for tracking runs and has model registry support. Official docs: https://mlflow.org/docs/latest/ml/tracking/ and https://mlflow.org/docs/latest/registry/

In [1]:
import pandas as pd
from datetime import datetime, timezone
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=900, n_features=14, n_informative=6, random_state=31)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=31)
runs=[]
for n_estimators, min_leaf in [(80,1),(160,1),(160,4)]:
    model=RandomForestClassifier(n_estimators=n_estimators,min_samples_leaf=min_leaf,random_state=31,n_jobs=1).fit(X_train,y_train)
    runs.append({"timestamp_utc": datetime.now(timezone.utc).isoformat(), "model": "RandomForest", "n_estimators":n_estimators,"min_samples_leaf":min_leaf,"roc_auc":roc_auc_score(y_test,model.predict_proba(X_test)[:,1]),"data_id":"synthetic-classification-v1","seed":31})
runs_df=pd.DataFrame(runs).sort_values("roc_auc",ascending=False)
display(runs_df.round(3))

,timestamp_utc,model,n_estimators,min_samples_leaf,roc_auc,data_id,seed
1,2026-10-19T11:19:34.816767+00:00,RandomForest,160,1,0.891,synthetic-classification-v1,31
0,2026-10-19T11:19:33.958847+00:00,RandomForest,80,1,0.889,synthetic-classification-v1,31
2,2026-10-19T11:19:35.536562+00:00,RandomForest,160,4,0.887,synthetic-classification-v1,31


## Optional MLflow logging pattern (not executed unless installed)
Install `mlflow` separately. Start with local tracking for learning, then use a secured shared tracking server for team workflows.

```python
# Optional snippet; install mlflow before running.
# import mlflow
# with mlflow.start_run():
#     mlflow.log_param("n_estimators", 160)
#     mlflow.log_param("min_samples_leaf", 4)
#     mlflow.log_metric("roc_auc", 0.91)  # replace with your measured value
#     mlflow.log_param("data_id", "dataset-version-or-URI")
```

Never log arbitrary credentials or sensitive raw data as run parameters or artifacts.

## A registry adds lifecycle metadata
A registry stores named models and versions with lineage, tags, annotations, and access policies. Teams commonly separate candidate, staging, and production releases. Versioning is not just file names: keep the code revision, data version, metrics, approvals, and deployment environment traceable.

## Practice
Design five fields that every experiment run in your project should record. Include at least one field about data and one about reproducibility.